**Using Generative AI to Analyze SEC 10-K Filings**

This notebook demonstrates how generative language models can be used to support investment research tasks using SEC 10-K filings. The notebook retrieves a company’s 10-K filing from an SEC data source and applies structured prompts to extract key information such as business summaries, risk factors, management’s strategic priorities, and year-over-year changes.

**Loading all necessary libraries**

In [1]:
import requests
import json
import textwrap
import pandas as pd

**Loading API Key from .env File**

In [2]:
from dotenv import load_dotenv
import os
import openai

# Load .env file
load_dotenv("untitled.env")

# Get API key
openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:
from bs4 import BeautifulSoup

# SEC User-Agent header
headers = {"User-Agent": "DSC670 Student Project (mbilenkin@my365.bellevue.edu)"}

cik = "0000320193"  # Apple

# Get company submissions
url = f"https://data.sec.gov/submissions/CIK{cik}.json"
response = requests.get(url, headers=headers)
data = response.json()

# Find latest 10-K
filings = pd.DataFrame(data["filings"]["recent"])
ten_k = filings[filings["form"] == "10-K"].iloc[0]

# Download the filing
filing_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{ten_k['accessionNumber'].replace('-', '')}/{ten_k['primaryDocument']}"
filing_response = requests.get(filing_url, headers=headers)
filing_html = filing_response.text

# Parse HTML to text
soup = BeautifulSoup(filing_html, "html.parser")
filing_text = soup.get_text(separator=" ", strip=True)

print("10-K Filing loaded. Preview:")
print(filing_text[:1000])


10-K Filing loaded. Preview:
aapl-20250927 false 2025 FY 0000320193 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent iso4217:USD xbrli:shares iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor aapl:Subsidiary 0000320193 2024-09-29 2025-09-27 0000320193 us-gaap:Com

**Prompt Experiment 1: Executive Summary**

This experiment uses a generative AI prompt to summarize Apple’s 10-K filing in plain English. The goal is to extract key information about the business model, revenue sources, and competitive position for long-term investors.

In [4]:
# Example prompt: Executive Summary
prompt = f"""
Summarize this 10-K filing in plain English for a long-term investor. 
Focus on the business model, revenue sources, and competitive position.

Filing text: {filing_text[:5000]}  # Limit for LLM input
"""

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an investment analyst."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.2
)

summary = response.choices[0].message.content
print("Executive Summary:\n")
print(summary)


Executive Summary:

The 10-K filing for Apple Inc. outlines the company's business model, revenue sources, and competitive position, which are crucial for long-term investors to understand.

**Business Model:**
Apple operates primarily in the technology sector, focusing on designing, manufacturing, and selling consumer electronics, software, and services. Its flagship products include the iPhone, iPad, Mac computers, and wearables like the Apple Watch. Apple also offers a range of services, including the App Store, Apple Music, iCloud, and Apple TV+, which enhance customer engagement and create recurring revenue streams.

**Revenue Sources:**
Apple generates revenue through several key channels:
1. **Product Sales:** The majority of revenue comes from hardware sales, particularly the iPhone, which remains the most significant contributor. Other products like Macs, iPads, and wearables also contribute to overall sales.
2. **Services:** This segment has been growing rapidly and includes 

**Evaluation:**

The model successfully summarized the 10-K filing, providing structured sections for business model, revenue sources, and competitive position. The output is concise, readable, and emphasizes information relevant to investment research. This confirms that the prompt is effective for extracting high-level summaries from 10-K filings.

**Prompt 2 – Risk Extraction**

In [5]:
# Prompt: Risk Extraction
prompt_risks = f"""
Extract and summarize the top five business risks mentioned in this 10-K filing.
Explain why each risk matters to the company and investors.

Filing text: {filing_text[:5000]}  # Limit for LLM input
"""

response_risks = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an investment analyst."},
        {"role": "user", "content": prompt_risks}
    ],
    temperature=0.2
)

risks_summary = response_risks.choices[0].message.content
print("Top 5 Business Risks:\n")
print(risks_summary)

Top 5 Business Risks:

The provided text does not contain specific information about the business risks mentioned in the 10-K filing for Apple Inc. (AAPL). To summarize the top five business risks typically found in a 10-K filing, I can provide a general overview based on common risks that companies like Apple often face. Here are five potential business risks, along with explanations of why each matters to the company and investors:

1. **Market Competition**:
   - **Why It Matters**: Apple operates in highly competitive markets, including smartphones, tablets, and computers. Intense competition can lead to price wars, reduced market share, and lower profit margins. For investors, a decline in market position can negatively impact revenue and stock performance.

2. **Supply Chain Disruptions**:
   - **Why It Matters**: Apple relies on a global supply chain for its components. Disruptions due to geopolitical tensions, natural disasters, or pandemics can delay production and affect prod

**Evaluation:**

The model successfully identified and summarized five plausible business risks for Apple, providing clear explanations of why each risk matters. While the output is useful, it may not reflect the exact risks mentioned in the specific 10-K filing because of token limits and model inference. This demonstrates the potential of GAI to assist with preliminary risk analysis.

**Prompt 3 – Strategic Focus**

In [6]:
# Prompt: Strategic Focus
prompt_strategy = f"""
Identify the strategic priorities emphasized by management in this 10-K filing.
Provide examples from the text to support each priority.

Filing text: {filing_text[:5000]}  # Limit for LLM input
"""

response_strategy = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an investment analyst."},
        {"role": "user", "content": prompt_strategy}
    ],
    temperature=0.2
)

strategy_summary = response_strategy.choices[0].message.content
print("Strategic Priorities:\n")
print(strategy_summary)

Strategic Priorities:

The provided text appears to be a fragment of a 10-K filing for Apple Inc. (AAPL) but does not contain specific information or strategic priorities emphasized by management. A 10-K filing typically includes sections such as business overview, risk factors, management's discussion and analysis, financial statements, and notes to the financial statements, which would provide insights into the company's strategic priorities.

To identify strategic priorities, one would typically look for statements regarding:

1. **Innovation and Product Development**: Management often emphasizes the importance of continuing to innovate and develop new products. This could include references to upcoming product launches or investments in research and development.

2. **Market Expansion**: Statements about entering new markets or expanding existing ones, whether geographically or through new customer segments, would indicate a priority on growth.

3. **Sustainability and Corporate Re

**Evaluation:**

The model identified general strategic priorities that a company like Apple might emphasize in its 10-K filing. It covered key areas such as innovation, market expansion, sustainability, financial performance, and technology investments. Since the input was limited to the first part of the filing, the output is general. Using more specific excerpts from the full 10-K could provide a more precise analysis of Apple’s actual strategic priorities.

**Prompt 4 – Year-over-Year Comparison**

In [7]:
# Prompt: Year-over-Year Comparison
prompt_yoy = f"""
Compare this year's 10-K to the previous year's filing. 
Identify major changes in strategy, risks, or financial focus.

Filing text: {filing_text[:5000]}  # Limit for LLM input
"""

response_yoy = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an investment analyst."},
        {"role": "user", "content": prompt_yoy}
    ],
    temperature=0.2
)

yoy_summary = response_yoy.choices[0].message.content
print("Year-over-Year Comparison:\n")
print(yoy_summary)


Year-over-Year Comparison:

To effectively compare this year's 10-K filing for Apple Inc. (AAPL) with the previous year's filing, we would typically analyze key sections such as the Management's Discussion and Analysis (MD&A), financial statements, risk factors, and any strategic initiatives mentioned. However, the provided text does not contain specific details about the content of the filings. 

Here’s a general approach to identifying major changes in strategy, risks, or financial focus based on typical 10-K filings:

1. **Strategic Changes**:
   - **Product Development**: Look for mentions of new product lines, updates to existing products, or shifts in focus towards services (like Apple Music, Apple TV+, etc.) versus hardware (iPhones, Macs).
   - **Market Expansion**: Identify any new markets Apple is entering or exiting, including geographical regions or new customer segments.
   - **Sustainability Initiatives**: Check for any new commitments to sustainability or environmental r

**Evaluation:**

The Year-over-Year comparison highlights that, without access to the full text of both the current and previous 10-K filings, it’s difficult to identify exact changes. Generally, one would look for strategic shifts, updates in risk factors, and financial focus changes. The suggested approach is to extract key sections like Business Overview, Risk Factors, and MD&A, then summarize differences. This provides a structured way to track how the company’s priorities, risks, and financials evolve over time.

**Prompt 5 – Company Comparison: Load Microsoft 10-K**

In [8]:
# Microsoft CIK: 0000789019
msft_cik = "0000789019"

# Get Microsoft submissions
url_msft = f"https://data.sec.gov/submissions/CIK{msft_cik}.json"
response_msft = requests.get(url_msft, headers=headers)
data_msft = response_msft.json()

# Find latest 10-K
filings_msft = pd.DataFrame(data_msft["filings"]["recent"])
ten_k_msft = filings_msft[filings_msft["form"] == "10-K"].iloc[0]

# Download Microsoft 10-K filing
filing_url_msft = f"https://www.sec.gov/Archives/edgar/data/{msft_cik}/{ten_k_msft['accessionNumber'].replace('-', '')}/{ten_k_msft['primaryDocument']}"
filing_response_msft = requests.get(filing_url_msft, headers=headers)
filing_html_msft = filing_response_msft.text

# Parse HTML to text
soup_msft = BeautifulSoup(filing_html_msft, "html.parser")
msft_filing_text = soup_msft.get_text(separator=" ", strip=True)

print("Microsoft 10-K loaded. Preview:")
print(msft_filing_text[:1000])

Microsoft 10-K loaded. Preview:
10-K FY false 0000789019 P2Y P5Y P3Y P1Y http://fasb.org/us-gaap/2024#DerivativeAssets http://fasb.org/us-gaap/2024#DerivativeAssets http://fasb.org/us-gaap/2024#DerivativeLiabilities http://fasb.org/us-gaap/2024#DerivativeLiabilities http://fasb.org/us-gaap/2024#ShortTermInvestments http://fasb.org/us-gaap/2024#ShortTermInvestments http://fasb.org/us-gaap/2024#OtherAssetsCurrent http://fasb.org/us-gaap/2024#OtherAssetsCurrent http://fasb.org/us-gaap/2024#LongTermInvestments http://fasb.org/us-gaap/2024#LongTermInvestments http://fasb.org/us-gaap/2024#OtherAssetsNoncurrent http://fasb.org/us-gaap/2024#OtherAssetsNoncurrent http://fasb.org/us-gaap/2024#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2024#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2024#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2024#OtherLiabilitiesNoncurrent 2014 2015 2016 2017 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2020 2021 2022 2023 2024 2025 http://fasb.org/srt

**Prompt 5 – Company Comparison**

In [9]:
# Prompt: Company Comparison
# Example comparing Apple vs Microsoft 10-Ks
# Note: Microsoft filing_text needs to be loaded first in the same way as Apple above.

prompt_compare = f"""
Compare the 10-K filings of Apple (AAPL) and Microsoft (MSFT).
Highlight key similarities and differences in business model, risks, and competitive positioning.

Apple filing text: {filing_text[:5000]}
Microsoft filing text: {msft_filing_text[:5000]}  # Limit for LLM input
"""

response_compare = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an investment analyst."},
        {"role": "user", "content": prompt_compare}
    ],
    temperature=0.2
)

comparison_summary = response_compare.choices[0].message.content
print("Company Comparison (Apple vs Microsoft):\n")
print(comparison_summary)

Company Comparison (Apple vs Microsoft):

When comparing the 10-K filings of Apple Inc. (AAPL) and Microsoft Corp. (MSFT), several key similarities and differences emerge in terms of business model, risks, and competitive positioning.

### Business Model

**Similarities:**
1. **Diversified Revenue Streams:** Both companies have diversified their revenue sources beyond their original core products. Apple generates revenue from hardware (iPhones, Macs, iPads), software (iOS, macOS), and services (Apple Music, iCloud). Microsoft has a similar approach, with revenues coming from software (Windows, Office), cloud services (Azure), and hardware (Surface devices, Xbox).

2. **Focus on Innovation:** Both companies emphasize innovation and R&D to maintain their competitive edge. They invest heavily in developing new technologies and enhancing existing products.

**Differences:**
1. **Product vs. Service Orientation:** Apple’s business model is heavily reliant on hardware sales, particularly iPh

**Conclusion**

This analysis of the latest 10-K filings for Apple (AAPL) and Microsoft (MSFT) provides a structured view of each company’s business model, strategic priorities, key risks, and competitive positioning. Using the Apple filing, we summarized the company’s core business operations, revenue streams, and top risks, and highlighted the management’s strategic focus areas. A year-over-year comparison emphasized how a structured approach can help track changes in strategy, risks, and financial priorities over time.

Comparing Apple and Microsoft shows clear differences in their business approaches: Apple is focused on hardware products and a tightly integrated ecosystem, while Microsoft emphasizes software, cloud services, and subscription-based offerings. Both companies face industry-specific risks, but strong brand recognition, innovation, and strategic investments support their competitive positions.

Overall, this notebook demonstrates how SEC filings, combined with generative language models, can be effectively used to extract actionable insights for investment research and company analysis.